In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [3]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime
from pathlib import Path  # 1. Importamos Pathlib

app = Flask(__name__)

# Configuración de rutas seguras y locales en tu área de trabajo
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOG_FOLDER = BASE_DIR / "logs"  # Creará la carpeta 'logs' dentro de tu carpeta actual

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE (CORREGIDO) ---
def log_event(message):
    # Crea la carpeta 'logs' de forma segura en tu espacio local
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    
    log_file = LOG_FOLDER / "local_chassis_history.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now()}] {message}")

# if __name__ == "__main__":
#     # Activamos debug=True para que si algo más falla, la Mac te diga exactamente qué línea es en lugar del error genérico
#     app.run(host="0.0.0.0", port=5000, debug=True)

if __name__ == "__main__":
    # Agregamos use_reloader=False para que no choque con Jupyter
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.20.10.3:5000
Press CTRL+C to quit
172.20.10.3 - - [26/Jun/2026 09:58:46] "GET / HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:01:23] "GET /diagnostics HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:01:36] "POST /clear-dtc HTTP/1.1" 302 -
172.20.10.3 - - [26/Jun/2026 10:01:36] "GET / HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:01:37] "POST /reset-dtc HTTP/1.1" 302 -
172.20.10.3 - - [26/Jun/2026 10:01:37] "GET / HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:01:38] "GET /diagnostics HTTP/1.1" 200 -


In [4]:
ab = """from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime
from pathlib import Path  # 1. Importamos Pathlib

app = Flask(__name__)

# Configuración de rutas seguras y locales en tu área de trabajo
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOG_FOLDER = BASE_DIR / "logs"  # Creará la carpeta 'logs' dentro de tu carpeta actual

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f'''
        <h2>📊 Data Retrieved From Cloud Bucket:</h2>
        <p>{cloud_data}</p>
        <br>
        <p><a href="/">⬅️ Return to Home Page</a></p>
        '''
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return f'''
        <h2>📊 Data Retrieved From Cloud Bucket:</h2>
        <p>✅ NO ACTIVE FAULT CODES</p>
        <br>
        <p><a href="/">⬅️ Return to Home Page</a></p>
        '''

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE (CORREGIDO) ---
def log_event(message):
    # Crea la carpeta 'logs' de forma segura en tu espacio local
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    
    log_file = LOG_FOLDER / "local_chassis_history.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    # Agregamos use_reloader=False para que no choque con Jupyter
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)"""

with open("app.py", "w") as f:
    f.write(ab)

print("🖥️ Dashboard logic compiled successfully into app.py!") 

🖥️ Dashboard logic compiled successfully into app.py!


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime
from pathlib import Path  # 1. Importamos Pathlib

app = Flask(__name__)

# Configuración de rutas seguras y locales en tu área de trabajo
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOG_FOLDER = BASE_DIR / "logs"  # Creará la carpeta 'logs' dentro de tu carpeta actual

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f'''
        <h2>📊 Data Retrieved From Cloud Bucket:</h2>
        <p>{cloud_data}</p>
        <br>
        <p><a href="/">⬅️ Return to Home Page</a></p>
        '''
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return f'''
        <h2>📊 Data Retrieved From Cloud Bucket:</h2>
        <p>✅ NO ACTIVE FAULT CODES</p>
        <br>
        <p><a href="/">⬅️ Return to Home Page</a></p>
        '''

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE (CORREGIDO) ---
def log_event(message):
    # Crea la carpeta 'logs' de forma segura en tu espacio local
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    
    log_file = LOG_FOLDER / "local_chassis_history.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    # Agregamos use_reloader=False para que no choque con Jupyter
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.20.10.3:5000
Press CTRL+C to quit
172.20.10.3 - - [26/Jun/2026 10:08:44] "GET / HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:08:48] "GET /diagnostics HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:08:52] "GET / HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:08:54] "GET /diagnostics HTTP/1.1" 200 -
172.20.10.3 - - [26/Jun/2026 10:08:55] "GET / HTTP/1.1" 200 -


In [ ]:
aaaaaaaaaaa

In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)            


In [ ]:

web_interactive_app = """from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

# Helper function to seed the data initially
def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

@app.route("/")
def home():
    # Check if the file still exists in our S3 cloud storage
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    # Create local log history entry
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] Dashboard loaded.\\n")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
        </body>
    </html>
    '''

@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    # Remove the file completely from the cloud storage bucket
    s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    # Put the code back so we can test it again
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    return redirect(url_for("home"))

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""

with open("app.py", "w") as f:
    f.write(web_interactive_app)

print("🖥️ Dashboard logic compiled successfully into app.py!")

In [ ]:
web_dual_interface_app = """from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""

with open("app.py", "w") as f:
    f.write(web_dual_interface_app)

print("🖥️ Dual-interface dashboard and data api successfully compiled into app.py!")

In [ ]:
!docker build -t diagnostic-cloud-app .

In [ ]:
# Writing the official assembly blueprint for our dual-interface app
dashboard_dockerfile = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 5000
CMD ["python", "app.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(dashboard_dockerfile)

print("🐳 Dockerfile successfully written to the landing zone!")

In [ ]:
# Re-verifying the exact package requirements manifest
final_requirements = """Flask==3.0.0
boto3
moto
"""

with open("requirements.txt", "w") as f:
    f.write(final_requirements)

print("📝 requirements.txt is written and verified!")

In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime

app = Flask(__name__)

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE ---
def log_event(message):
    log_folder = "/app/logs"
    if not os.path.exists(log_folder):
        os.makedirs(log_folder)
    with open(os.path.join(log_folder, "local_chassis_history.txt"), "a") as f:
        f.write(f"[{datetime.now()}] {message}\n")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


In [ ]:
!docker build --no-cache -t diagnostic-cloud-app .

In [ ]:
!docker run --rm -p 5000:5000 -v $(pwd)/my_local_logs:/app/logs diagnostic-cloud-app

In [ ]:
!docker kill $(docker ps -q) 2>/dev/null || true

In [ ]:
!docker run --rm -p 5000:5000 -v $(pwd)/my_local_logs:/app/logs diagnostic-cloud-app

In [ ]:
ab = """from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime
from pathlib import Path  # 1. Importamos Pathlib

app = Flask(__name__)

# Configuración de rutas seguras y locales en tu área de trabajo
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOG_FOLDER = BASE_DIR / "logs"  # Creará la carpeta 'logs' dentro de tu carpeta actual

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE (CORREGIDO) ---
def log_event(message):
    # Crea la carpeta 'logs' de forma segura en tu espacio local
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    
    log_file = LOG_FOLDER / "local_chassis_history.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now()}] {message}\n")

# if __name__ == "__main__":
#     # Activamos debug=True para que si algo más falla, la Mac te diga exactamente qué línea es en lugar del error genérico
#     app.run(host="0.0.0.0", port=5000, debug=True)

if __name__ == "__main__":
    # Agregamos use_reloader=False para que no choque con Jupyter
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)"""


with open("app.py", "w") as f:
    f.write(ab)

print("🖥️ Dual-interface dashboard and data api successfully compiled into app.py!")

In [ ]:
from flask import Flask, redirect, url_for
import boto3
from moto import mock_aws
import os
from datetime import datetime
from pathlib import Path  # 1. Importamos Pathlib

app = Flask(__name__)

# Configuración de rutas seguras y locales en tu área de trabajo
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOG_FOLDER = BASE_DIR / "logs"  # Creará la carpeta 'logs' dentro de tu carpeta actual

# Start AWS Simulation
mock = mock_aws()
mock.start()

s3 = boto3.client("s3", region_name="us-east-1")
BUCKET_NAME = "vehicle-diagnostic-vault"
FILE_NAME = "live_fault_code.txt"

def seed_bucket():
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=FILE_NAME,
        Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)"
    )

seed_bucket()

# --- 1. THE INTERACTIVE COCKPIT (HUMAN INTERFACE) ---
@app.route("/")
def home():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        status_html = f"<div style='color: red; font-weight: bold;'>{cloud_data}</div>"
        button_html = f"<form action='/clear-dtc' method='POST'><button style='padding: 10px; background-color: orange;'>🔧 Clear DTC (Reset System)</button></form>"
    except Exception:
        status_html = "<div style='color: green; font-weight: bold;'>✅ SYSTEM CLEAR - No Active Fault Codes Found.</div>"
        button_html = f"<form action='/reset-dtc' method='POST'><button style='padding: 10px; background-color: blue; color: white;'>⚡ Reload Fault Code</button></form>"

    log_event("Dashboard Home Page loaded.")

    return f'''
    <html>
        <body style="font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9;">
            <h1>🌐 Moby Bunny Cloud-Diagnostic Cockpit</h1>
            <hr>
            <h3>Current Vehicle System Status:</h3>
            {status_html}
            <br><br>
            {button_html}
            <br><br>
            <p><a href="/diagnostics">👉 Go to Raw Diagnostics Data Page</a></p>
        </body>
    </html>
    '''

# --- 2. THE RAW DIAGNOSTICS PAGE (DATA INTERFACE) ---
@app.route("/diagnostics")
def diagnostics():
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        cloud_data = response["Body"].read().decode("utf-8")
        log_event("Raw Diagnostics Route Access: Data Found.")
        return f"<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>{cloud_data}</p>"
    except Exception:
        log_event("Raw Diagnostics Route Access: System Clear.")
        return "<h2>📊 Data Retrieved From Cloud Bucket:</h2><p>✅ NO ACTIVE FAULT CODES</p>"

# --- ACTION ENDPOINTS ---
@app.route("/clear-dtc", methods=["POST"])
def clear_dtc():
    try:
        s3.delete_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
        log_event("Action: DTC Cleared from S3 Bucket.")
    except Exception:
        pass
    return redirect(url_for("home"))

@app.route("/reset-dtc", methods=["POST"])
def reset_dtc():
    s3.put_object(Bucket=BUCKET_NAME, Key=FILE_NAME, Body="🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)")
    log_event("Action: DTC Reloaded into S3 Bucket.")
    return redirect(url_for("home"))

# --- REUSABLE LOCAL CHASSIS LOG VALVE (CORREGIDO) ---
def log_event(message):
    # Crea la carpeta 'logs' de forma segura en tu espacio local
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    
    log_file = LOG_FOLDER / "local_chassis_history.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now()}] {message}\n")

# if __name__ == "__main__":
#     # Activamos debug=True para que si algo más falla, la Mac te diga exactamente qué línea es en lugar del error genérico
#     app.run(host="0.0.0.0", port=5000, debug=True)

if __name__ == "__main__":
    # Agregamos use_reloader=False para que no choque con Jupyter
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)